# MultiTaxi benchmark

Run this notebook from the project's `app/` environment. One versioned JSON file stores each run, its final evaluation, and its convergence checkpoints.

In [ ]:
import json
import os
import time
from dataclasses import asdict, dataclass

import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from scripts import train_dqn_agent, train_hrm_agent, train_qt
from src.config import Configuration
from src.models import evaluate_agent


In [ ]:
@dataclass
class BenchmarkConfiguration(Configuration):
    training_seeds: tuple[int, ...] = tuple(range(42, 52))
    convergence_interval: int = 1000
    rerun_completed: bool = False
    record_version: int = 4


BENCHMARK = BenchmarkConfiguration(
    n_training_episodes=200000,
    n_eval_episodes=200,
    eval_seed_base=20260720,
    multitaxi_grid_size=10,
    video_fps=10,
)
TRAINING_SEEDS = list(BENCHMARK.training_seeds)
EVALUATION_SEEDS = BENCHMARK.eval_seed
if not TRAINING_SEEDS:
    raise ValueError('training_seeds cannot be empty')
if BENCHMARK.convergence_interval <= 0:
    raise ValueError('convergence_interval must be positive')

VARIANTS = (
    {'id': 'qlearning', 'name': 'Q-learning', 'kind': 'qtable', 'config': 'qlearning.yaml'},
    {'id': 'qrm', 'name': 'QRM', 'kind': 'qtable', 'config': 'qrm.yaml'},
    {'id': 'qrm_crm', 'name': 'QRM + CRM', 'kind': 'qtable', 'config': 'qrm_crm.yaml'},
    {'id': 'hrm', 'name': 'HRM', 'kind': 'hrm', 'config': 'hrm.yaml'},
    # {'id': 'dqn', 'name': 'DQN', 'kind': 'dqn', 'config': 'dqn_plain.yaml'},
    # {'id': 'dqn_rm', 'name': 'DQN + RM', 'kind': 'dqn', 'config': 'dqn_rm.yaml'},
    # {'id': 'dqn_rm_crm', 'name': 'DQN + RM + CRM', 'kind': 'dqn', 'config': 'dqn.yaml'},
)
VARIANTS_BY_ID = {variant['id']: variant for variant in VARIANTS}
RESULTS_PATH = os.path.join(
    BENCHMARK.DATA_PATH,
    f'multitaxi_{BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size}_benchmark_v{BENCHMARK.record_version}.json',
)

print(f'Runs: {len(VARIANTS) * len(TRAINING_SEEDS)}')
print(f'Training episodes per run: {BENCHMARK.n_training_episodes}')
print(f'Evaluation episodes per run: {len(EVALUATION_SEEDS)}')
print(f'Results: {RESULTS_PATH}')


In [ ]:
METRIC_KEYS = (
    'successes', 'episodes', 'invalid_actions', 'mean_reward', 'reward_std',
    'successful_std', 'mean_successful_steps', 'worst_reward',
)


def benchmark_config(seed, variant):
    config = Configuration(yaml_config_path=variant['config'])
    config.set_seed(seed)
    config.exp_name = variant['id']
    config.multitaxi_grid_size = BENCHMARK.multitaxi_grid_size
    config.video_fps = BENCHMARK.video_fps
    config.n_training_episodes = BENCHMARK.n_training_episodes
    config.n_eval_episodes = BENCHMARK.n_eval_episodes
    config.eval_seed = EVALUATION_SEEDS
    return config


def variant_spec(variant):
    config = benchmark_config(BENCHMARK.seed, variant)
    config_path = os.path.join(config.CONFIGS_PATH, variant['config'])
    with open(config_path, encoding='utf-8') as file:
        yaml = file.read()
    return {
        'id': variant['id'],
        'name': variant['name'],
        'kind': variant['kind'],
        'config': variant['config'],
        'use_rm': config.use_rm,
        'use_crm': config.use_crm,
        'observation': config.multitaxi_observation_mode,
        'yaml': yaml,
        'resolved': {
            key: value
            for key, value in asdict(config).items()
            if key not in {'CONFIGS_PATH', 'DATA_PATH', 'MODELS_PATH', 'LOGS_PATH', 'VIDEO_PATH', 'eval_seed', 'parse_state', 'seed', 'yaml_config_path'}
        },
    }


BENCHMARK_SPEC = {
    'version': BENCHMARK.record_version,
    'grid_size': BENCHMARK.multitaxi_grid_size,
    'training_episodes': BENCHMARK.n_training_episodes,
    'evaluation_episodes': BENCHMARK.n_eval_episodes,
    'training_seeds': TRAINING_SEEDS,
    'evaluation_seeds': EVALUATION_SEEDS,
    'video_fps': BENCHMARK.video_fps,
    'convergence_interval': BENCHMARK.convergence_interval,
    'variants': [variant_spec(variant) for variant in VARIANTS],
}


def normalize_metrics(metrics):
    normalized = {}
    for key in METRIC_KEYS:
        value = metrics[key]
        if isinstance(value, np.generic):
            value = value.item()
        normalized[key] = None if isinstance(value, float) and not np.isfinite(value) else value
    return normalized


def validate_metrics(metrics):
    if not isinstance(metrics, dict) or not set(METRIC_KEYS) <= metrics.keys():
        raise ValueError('Evaluation metrics do not match the experiment schema')
    for key, value in metrics.items():
        if value is None and key in {'successful_std', 'mean_successful_steps'}:
            continue
        if not isinstance(value, (int, float)) or not np.isfinite(value):
            raise ValueError('Evaluation metrics do not match the experiment schema')


def validate_run(run):
    required = {'variant_id', 'variant', 'kind', 'config', 'seed', 'pipeline_seconds', 'metrics', 'convergence'}
    if not isinstance(run, dict) or not required <= run.keys():
        raise ValueError('Experiment runs do not match the v4 schema')
    variant = VARIANTS_BY_ID.get(run['variant_id'])
    if (
        variant is None
        or run['variant'] != variant['name']
        or run['kind'] != variant['kind']
        or run['config'] != variant['config']
        or not isinstance(run['seed'], int)
        or run['seed'] not in TRAINING_SEEDS
        or not isinstance(run['convergence'], list)
    ):
        raise ValueError('Experiment runs do not match the v4 schema')
    if run['metrics'] is None:
        if run['pipeline_seconds'] is not None:
            raise ValueError('Incomplete experiment runs cannot have a duration')
    else:
        validate_metrics(run['metrics'])
        if not isinstance(run['pipeline_seconds'], (int, float)) or run['pipeline_seconds'] < 0:
            raise ValueError('Experiment runs do not match the v4 schema')
    for checkpoint in run['convergence']:
        if (
            not isinstance(checkpoint, dict)
            or not isinstance(checkpoint.get('episode'), int)
            or not 0 < checkpoint['episode'] <= BENCHMARK.n_training_episodes
        ):
            raise ValueError('Convergence checkpoints do not match the v4 schema')
        validate_metrics(checkpoint.get('metrics'))
    if len({checkpoint['episode'] for checkpoint in run['convergence']}) != len(run['convergence']):
        raise ValueError('Experiment runs contain duplicate convergence checkpoints')


def load_runs():
    if not os.path.exists(RESULTS_PATH):
        return []
    with open(RESULTS_PATH, encoding='utf-8') as file:
        payload = json.load(file)
    if not isinstance(payload, dict) or payload.get('spec') != BENCHMARK_SPEC:
        raise ValueError(f'Results at {RESULTS_PATH} do not match the current benchmark specification.')
    runs = payload.get('runs')
    if not isinstance(runs, list):
        raise ValueError('Experiment runs must be a list')
    for run in runs:
        validate_run(run)
    if len({(run['variant_id'], run['seed']) for run in runs}) != len(runs):
        raise ValueError('Experiment records contain duplicate runs')
    return runs


def save_runs(runs):
    temporary_path = f'{RESULTS_PATH}.tmp'
    with open(temporary_path, 'w', encoding='utf-8') as file:
        json.dump({'spec': BENCHMARK_SPEC, 'runs': runs}, file, indent=2, allow_nan=False)
    os.replace(temporary_path, RESULTS_PATH)


def completed_run(run):
    return run['metrics'] is not None and any(
        checkpoint['episode'] == BENCHMARK.n_training_episodes
        for checkpoint in run['convergence']
    )


RUNNERS = {
    'qtable': train_qt,
    'hrm': train_hrm_agent,
    'dqn': train_dqn_agent,
}


In [ ]:
runs = load_runs()
for variant in VARIANTS:
    for seed in TRAINING_SEEDS:
        run_id = (variant['id'], seed)
        existing_run = next((run for run in runs if (run['variant_id'], run['seed']) == run_id), None)
        video_path = os.path.join(
            BENCHMARK.VIDEO_PATH,
            f"{BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size}_{variant['id']}_seed{seed}_video.gif",
        )
        if existing_run and completed_run(existing_run) and os.path.isfile(video_path) and not BENCHMARK.rerun_completed:
            print(f'Skipping {variant["name"]}, seed {seed}')
            continue

        run = {
            'variant_id': variant['id'],
            'variant': variant['name'],
            'kind': variant['kind'],
            'config': variant['config'],
            'seed': seed,
            'pipeline_seconds': None,
            'metrics': None,
            'convergence': [],
        }
        if existing_run is None:
            runs.append(run)
            save_runs(runs)
        config = benchmark_config(seed, variant)

        def progress_callback(episode, agent, env, get_propositions):
            if episode % BENCHMARK.convergence_interval and episode != BENCHMARK.n_training_episodes:
                return
            metrics = normalize_metrics(evaluate_agent(
                config, agent, get_propositions, env,
                seeds=EVALUATION_SEEDS, report=False, return_metrics=True,
            ))
            run['convergence'].append({'episode': episode, 'metrics': metrics})
            if existing_run is None:
                save_runs(runs)

        started = time.perf_counter()
        RUNNERS[variant['kind']](config, progress_callback=progress_callback)
        final_checkpoint = next(
            (checkpoint for checkpoint in run['convergence'] if checkpoint['episode'] == BENCHMARK.n_training_episodes),
            None,
        )
        if final_checkpoint is None:
            raise RuntimeError('Training finished without a final evaluation checkpoint')
        run['pipeline_seconds'] = time.perf_counter() - started
        run['metrics'] = final_checkpoint['metrics']
        runs = [stored_run for stored_run in runs if (stored_run['variant_id'], stored_run['seed']) != run_id]
        runs.append(run)
        save_runs(runs)
        print(
            f"{variant['name']}, seed {seed}: {run['metrics']['mean_reward']:.2f} reward, "
            f"{run['metrics']['mean_successful_steps']} mean successful steps"
        )


In [ ]:
runs = [run for run in load_runs() if completed_run(run)]
if not runs:
    raise ValueError(f'No completed benchmark runs found in {RESULTS_PATH}')

summary = []
for variant in VARIANTS:
    variant_runs = [run for run in runs if run['variant_id'] == variant['id']]
    if not variant_runs:
        continue
    rewards = np.asarray([run['metrics']['mean_reward'] for run in variant_runs])
    steps = np.asarray([
        run['metrics']['mean_successful_steps']
        for run in variant_runs
        if run['metrics']['mean_successful_steps'] is not None
    ])
    successes = sum(run['metrics']['successes'] for run in variant_runs)
    episodes = sum(run['metrics']['episodes'] for run in variant_runs)
    summary.append({
        'variant': variant['name'],
        'runs': len(variant_runs),
        'mean_reward': float(rewards.mean()),
        'reward_std_across_runs': float(rewards.std()),
        'success_rate': successes / episodes,
        'mean_successful_steps': float(steps.mean()) if len(steps) else None,
    })

display(summary)


In [ ]:
figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Final evaluation reward', 'Steps to solve successful episodes'),
)
for variant in VARIANTS:
    variant_runs = [run for run in runs if run['variant_id'] == variant['id']]
    rewards = [run['metrics']['mean_reward'] for run in variant_runs]
    steps = [
        run['metrics']['mean_successful_steps']
        for run in variant_runs
        if run['metrics']['mean_successful_steps'] is not None
    ]
    figure.add_trace(go.Box(y=rewards, name=variant['name'], boxpoints='all'), row=1, col=1)
    if steps:
        figure.add_trace(go.Box(y=steps, name=variant['name'], boxpoints='all', showlegend=False), row=1, col=2)

figure.update_yaxes(title_text='Mean environment reward', row=1, col=1)
figure.update_yaxes(title_text='Mean steps', row=1, col=2)
figure.update_layout(
    title=f'MultiTaxi {BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size} final evaluation',
    template='plotly_white', height=550,
)
figure.show()


In [ ]:
convergence_figure = go.Figure()
for variant in VARIANTS:
    checkpoints = [
        (run['seed'], checkpoint)
        for run in runs
        if run['variant_id'] == variant['id']
        for checkpoint in run['convergence']
    ]
    values_by_episode = {}
    for seed, checkpoint in checkpoints:
        values_by_episode.setdefault(checkpoint['episode'], {})[seed] = checkpoint['metrics']['mean_reward']
    episodes = [
        episode for episode, values in sorted(values_by_episode.items())
        if set(values) == set(TRAINING_SEEDS)
    ]
    if not episodes:
        continue
    rewards = [list(values_by_episode[episode].values()) for episode in episodes]
    convergence_figure.add_trace(go.Scatter(
        x=episodes,
        y=[np.mean(values) for values in rewards],
        error_y={'type': 'data', 'array': [np.std(values) for values in rewards], 'visible': True},
        mode='lines+markers',
        name=variant['name'],
    ))

convergence_figure.update_layout(
    title=f'MultiTaxi {BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size} reward convergence',
    xaxis_title='Training episodes',
    yaxis_title='Mean environment reward',
    template='plotly_white', height=600,
)
convergence_figure.show()
